In [46]:
import numpy as np
from numpy.typing import NDArray
from qiskit import QuantumCircuit
from qiskit.primitives import BaseSamplerV2
from qiskit.quantum_info import SparsePauliOp

from advanced_estimation.commutation.general_commuting import GeneralCommutation
from advanced_estimation.estimation.pauli_estimation import (
    estimate_cliques_expectation_values_and_covariances,
    overall_paulis_expectation_values_and_covariances,
)

In [47]:
# Define the observable as a sum of Pauli operators
# Group 1: XX and YY (commute - they have 2 anti-commutations: XY and YX)
# Group 2: ZZ and IZ (commute - bitwise commutation)
observable = SparsePauliOp(
    ["XX", "YY", "ZZ", "IZ"],
    coeffs=[0.5, -0.5, 1.0, 0.2]
)

# Quantum circuit that prepares the initial state
state_circuit = QuantumCircuit(2)
state_circuit.h(0)
print(f"State preparation circuit:\n{state_circuit}\n")

# Commutation module for finding commuting groups
commutation_module = GeneralCommutation()

# Total number of measurement shots for estimation
shots_budget = 1000

State preparation circuit:
     ┌───┐
q_0: ┤ H ├
     └───┘
q_1: ─────
          



## Step 1: Identify Commuting Pauli Groups

Let's start by determining which Pauli strings commute with each other and group them into cliques.

In [51]:
paulis = observable.paulis

cliques_paulis_indices = commutation_module.find_commuting_cliques(paulis)
print(f"\nFound {len(cliques_paulis_indices)} commuting clique(s):\n")
for clique_idx, clique in enumerate(cliques_paulis_indices):
    pauli_strings = [str(paulis[i]) for i in clique]
    print(f"  Clique {clique_idx + 1}: indices {clique} → {pauli_strings}")


Found 2 commuting clique(s):

  Clique 1: indices [2 0 1] → ['ZZ', 'XX', 'YY']
  Clique 2: indices [2 3] → ['ZZ', 'IZ']


## Step 2: Allocate Measurement Shots

Distribute an equal number of shots to each clique.

In [53]:
num_cliques = len(cliques_paulis_indices)
cliques_shots = [max(shots_budget // num_cliques, 1) for _ in range(num_cliques)]

print(f"Shot allocation across {num_cliques} clique(s):")
for clique_idx, shots in enumerate(cliques_shots):
    print(f"  Clique {clique_idx + 1}: {shots} shots")
print(f"\nTotal shots allocated: {sum(cliques_shots)} (budget: {shots_budget})")

Shot allocation across 2 clique(s):
  Clique 1: 500 shots
  Clique 2: 500 shots

Total shots allocated: 1000 (budget: 1000)
